# Part I - XML and HTML Data

In [14]:
#dependancies
import pandas as pd
import xml.etree.cElementTree as ET #XML Parser
from lxml import etree #ElementTree and lxml allow us to parse the XML file.
import requests #make request to server
import time #pause loop
from bs4 import BeautifulSoup

## Web Scraping

First we will go through how to parse one XML file. The Old Bailey API has a total of **197751** cases. Fortunately, we are only going to use the ones from 1843-1913, but that still only narrows the number of cases to somewhere above 77248! 

Don't worry though, you're not going to manually download each case yourself. This is where web scraping comes into play. With web scraping, we can automate data collection to get all the cases. 

Before we start scraping, we need to know how `requests` works. The `requests` library gets (`.get`) you a response object from a web server and will automatically decode the content from the server, from which you can use `.json()` to see the document! Requests through the Old Bailey API will return a dictionary, embedded in which is the XML representation of the trial account, which we can then write as a file and save.


In [15]:
print("We are using",1913-1843,"years of data to make classification")

We are using 70 years of data to make classification


In [16]:
url = "https://www.dhi.ac.uk/api/data/oldbailey_record"        # Base URL for the Old Bailey API endpoint
start_yr = 1902
end_yr = 1913

params = {"year_gte": start_yr,      
          "year_lte": end_yr }

headers = {"Accept": "application/json",     
           "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

r = requests.get(url,            
                 params=params,  
                 headers=headers)
r.raise_for_status()

trials = r.json()

In [17]:
trials

{'took': 4,
 'timed_out': False,
 '_shards': {'total': 5, 'successful': 5, 'skipped': 0, 'failed': 0},
 'hits': {'total': 9192,
  'max_score': None,
  'hits': [{'_index': 'dhids_oldbailey_record',
    '_type': 'doc',
    '_id': 'AYxELn6sPld_y58RzxIm',
    '_score': None,
    '_source': {'idkey': 'f19020113',
     'images': ['https://www.dhi.ac.uk/san/ccc/19020113/190201130001.jpg',
      'https://www.dhi.ac.uk/san/ccc/19020113/190201130002.jpg',
      'https://www.dhi.ac.uk/san/ccc/19020113/190201130003.jpg'],
     'text': "CENTRAL CRIMINAL COURT Sessions Paper. DIMSDALE, MAYOR. THIRD SESSION, HELD JANUARY 13TH, 1902. MINUTES OF EVIDENCE, TAKEN IN SHORT-HAND BY JAMES DROVER BARNETT AND ALEXANDER BUCKLER , Short-hand Writers to the Court, ROLLS CHAMBERS , No. 89, CHANCERY LANE. LONDON: STEVENS AND SONS, LIMITED, 119, CHANCERY LANE. Law Booksellers and Publishers. THE WHOLE PROCEEDINGS On the King's Commission of OYER AND TERMINER AND GAOL DELIVERY FOR The City of London, AND GAOL DELIVE

In [18]:
total_hits = trials['hits']['total']
total_hits

9192

In [19]:
# Define our  initial variables
xml_list = [] # This list will store all trial records collected across multiple API requests
i = 0            # Offset counter used for pagination (the API returns results in batches of 10)

# Get the total count dynamically so the loop knows when to stop
url = "https://www.dhi.ac.uk/api/data/oldbailey_record"

params = {'year_gte': start_yr, 
          'year_lte': end_yr}

headers = {"User-Agent": "Mozilla/5.0"} # Prevents 403 errors

probe = requests.get(url, 
                     params=params, 
                     headers=headers).json()

total_hits = probe['hits']['total']  # 1312

# Get the xmls
while i < total_hits:                                  # Continue requesting data until we have fetched all available trials
    query_url = f'{url}?from={i}&size=10&year_gte={start_yr}&year_lte={end_yr}' # Construct the API request URL with pagination (from/size) and year filters
    try:                                           # Send the GET request to the server using the browser-mimicking headers defined earlier
        response = requests.get(query_url,  
                                headers=headers)
        response.raise_for_status()                 # Raise an error if the request failed (e.g., 403 Forbidden or 500 Server Error)
        trials = response.json()                    # Parse the JSON response body into a Python dictionary
        batch = trials['hits']['hits']              # Extract the list of records (the "hits") from the nested JSON structure
        if not batch:                              # Safety break if the server returns empty results
            break 
            
        xml_list.extend(batch)                                        # Add the 10 records from this batch into our master list (xml_list)
        if len(xml_list) % 100 == 0 or len(xml_list) >= total_hits:    # Logic to print progress every 100 records - Provide a visual update
            print(f"Progress: {len(xml_list)} / {total_hits}")
            
        i += 10         # Increment the offset by 10 to "turn the page" for the next API request
        time.sleep(0.1) # Pause for 0.1 seconds to avoid triggering the server's anti-bot/rate-limiting protections
        
    except Exception as e:     # If any error occurs (network issue, JSON error, etc.), print the error and stop
        print(f"Error at offset {i}: {e}")
        break

Progress: 100 / 9192
Progress: 200 / 9192
Progress: 300 / 9192
Progress: 400 / 9192
Progress: 500 / 9192
Progress: 600 / 9192
Progress: 700 / 9192
Progress: 800 / 9192
Progress: 900 / 9192
Progress: 1000 / 9192
Progress: 1100 / 9192
Progress: 1200 / 9192
Progress: 1300 / 9192
Progress: 1400 / 9192
Progress: 1500 / 9192
Progress: 1600 / 9192
Progress: 1700 / 9192
Progress: 1800 / 9192
Progress: 1900 / 9192
Progress: 2000 / 9192
Progress: 2100 / 9192
Progress: 2200 / 9192
Progress: 2300 / 9192
Progress: 2400 / 9192
Progress: 2500 / 9192
Progress: 2600 / 9192
Progress: 2700 / 9192
Progress: 2800 / 9192
Progress: 2900 / 9192
Progress: 3000 / 9192
Progress: 3100 / 9192
Progress: 3200 / 9192
Progress: 3300 / 9192
Progress: 3400 / 9192
Progress: 3500 / 9192
Progress: 3600 / 9192
Progress: 3700 / 9192
Progress: 3800 / 9192
Progress: 3900 / 9192
Progress: 4000 / 9192
Progress: 4100 / 9192
Progress: 4200 / 9192
Progress: 4300 / 9192
Progress: 4400 / 9192
Progress: 4500 / 9192
Progress: 4600 / 91

In [20]:
xml_list[-10:] # last 10 entries - to check if we got the 1756

[{'_index': 'dhids_oldbailey_record',
  '_type': 'doc',
  '_id': 'AYxENIxsPld_y58R0WE4',
  '_score': None,
  '_source': {'idkey': 't19130401-58',
   'images': ['https://www.dhi.ac.uk/san/ccc/19130401/191304010097.jpg'],
   'text': 'COVERDALE, Victor Stanley, otherwise Thomas Rathbourne (36, surveyor) , obtaining by false pretences from Gerald Offley Forrester the several sums of £50 and £50, from Henry Brown Gold £100, from Henry Walker £100, from William Alfred McCarthy £20, and from James Francis Blount Dinwhiddie £100, in each case with intent to defraud, and unlawfully obtaining credit from the said several persons by means of false pretences and other fraud, and being an undischarged bankrupt unlawfully obtainin',
   'title': 'VICTOR STANLEY COVERDALE. Deception; fraud. 1st April 1913.'},
  'sort': [-1790985600000, 58]},
 {'_index': 'dhids_oldbailey_record',
  '_type': 'doc',
  '_id': 'AYxENIxsPld_y58R0WE5',
  '_score': None,
  '_source': {'idkey': 't19130401-59',
   'images': ['h

In [21]:
# inspect the first trial xml in the list
xml_list[0]

{'_index': 'dhids_oldbailey_record',
 '_type': 'doc',
 '_id': 'AYxELn6sPld_y58RzxIm',
 '_score': None,
 '_source': {'idkey': 'f19020113',
  'images': ['https://www.dhi.ac.uk/san/ccc/19020113/190201130001.jpg',
   'https://www.dhi.ac.uk/san/ccc/19020113/190201130002.jpg',
   'https://www.dhi.ac.uk/san/ccc/19020113/190201130003.jpg'],
  'text': "CENTRAL CRIMINAL COURT Sessions Paper. DIMSDALE, MAYOR. THIRD SESSION, HELD JANUARY 13TH, 1902. MINUTES OF EVIDENCE, TAKEN IN SHORT-HAND BY JAMES DROVER BARNETT AND ALEXANDER BUCKLER , Short-hand Writers to the Court, ROLLS CHAMBERS , No. 89, CHANCERY LANE. LONDON: STEVENS AND SONS, LIMITED, 119, CHANCERY LANE. Law Booksellers and Publishers. THE WHOLE PROCEEDINGS On the King's Commission of OYER AND TERMINER AND GAOL DELIVERY FOR The City of London, AND GAOL DELIVERY FOR THE COUNTY OF MIDDLESEX",
  'title': 'Front matter. 13th January 1902.'},
 'sort': [-2144880000000, 0]}

Now we can see clearly what the requests call got us. It is a list of dictionaries that has various keys, including 'idkey' for the document ID, 'images' for the page images of the paper Old Bailey Proceeds, and 'text', which shows **just the plain text** for the document (which is incomplete). Notice the texts tend to end mid-sentence.

We did **not** get the XML representation of the trial record, which has tags and labels for the parties etc., as we saw when we got the list of terms above. 

What we need to do next is
* get the list of ID keys for all 19225 documents from 1890 to 1913 (some of these will not be trial session records but will be front matter or advertising--we won't worry about that now)
* use the list of individual ID keys to make requests for the xml files for each ID key
* write the xml files to the local data directory
* use those xml files to build a dataframe

First we need to pick out the document ID from each item in the list.

In [22]:
xml_list[1001]['_source']['idkey']

't19030309-312'

In [23]:
# Now, let's get a list of document IDs we can work with. Then we can use the first ten to demonstrate the next step of calling the API to get the xml file for each document ID.
doc_ids = [xml_list[i]['_source']['idkey'] for i in range(0, len(xml_list))]
first_10 = doc_ids[:10]
first_10

['f19020113',
 't19020113-105',
 't19020113-2',
 't19020113-3',
 't19020113-4',
 't19020113-5',
 't19020113-6',
 't19020113-7',
 't19020113-112',
 't19020113-113']

Using the trial IDs from the previous cell, we are going to format the URL in a way so that we can get the XML file for each trial. In order to get the XML file using the Old Bailey API, we must follow this URL format:

<p style="text-align: center;">`https://www.dhi.ac.uk/api/data/oldbailey_record_single?idkey=`  </p>

For example, https://www.dhi.ac.uk/api/data/oldbailey_record_single?idkey=t16740429-1 gives you the link to the XML file of the first proceeding in the database.


In [24]:
fifth_trial_id = first_10[4]
fifth_trial_id

't19020113-4'

In [25]:
url = f'https://www.dhi.ac.uk/api/data/oldbailey_record_single?idkey={fifth_trial_id}'

#  Make the request with the headers
response = requests.get(url, 
                        headers=headers)

In [26]:
response.json()

{'took': 8,
 'timed_out': False,
 '_shards': {'total': 20, 'successful': 20, 'skipped': 0, 'failed': 0},
 'hits': {'total': 1,
  'max_score': 10.212662,
  'hits': [{'_index': 'dhids_oldbailey_record',
    '_type': 'doc',
    '_id': 'AYxELn6sPld_y58RzxIq',
    '_score': 10.212662,
    '_source': {'metadata': '<table xmlns:dhids="https://www.dhi.ac.uk/data/" class="table small"><tbody><tr><th scope="row">Text type</th><td>Trial account</td></tr><tr><th scope="row">Defendants</th><td>WALTER LACEY</td></tr><tr><th scope="row">Offences</th><td><a href="../about/crimes#deception">Deception</a> &gt; <a href="../about/crimes#forgery">Forgery</a></td></tr><tr><th scope="row">Session Date</th><td><a href="t19020113#t19020113-4">13th January 1902</a></td></tr><tr><th scope="row">Reference Number</th><td>t19020113-4</td></tr><tr><th scope="row">Verdicts</th><td><a href="../about/verdicts#guilty">Guilty</a> &gt; <a href="../about/verdicts#pleadedguilty">Pleaded guilty</a></td></tr><tr><th scope="ro

In [27]:
response.json()['hits']['hits'][0]['_source']['xml']

'<div1 type="trialAccount" id="t19020113-4"> <interp inst="t19020113-4" type="prevdiv" value="t19020113-3" divtype="trialAccount"/> <interp inst="t19020113-4" type="nextdiv" value="t19020113-5" divtype="trialAccount"/> <interp inst="t19020113-4" type="div0" value="t19020113" divtype="sessionsPaper"/> <interp inst="t19020113-4" type="uri" value="sessionsPapers/19020113"/> <interp inst="t19020113-4" type="date" value="19020113"/> <xptr imgpath="ccc/19020113/190201130003.jpg" imgtitle="Proceedings of the Central Criminal Court, 13th January ." imgrights="Copyright in this image is owned by the \'Old Bailey Online\' project. Non-commercial and fair use of this image is allowed without further consent. Commercial use is prohibited without explicit permission from the project." type="pageFacsimile" value="preceding" doc="190201130003"/> <join result="criminalCharge" id="t19020113-4-charge-1" targOrder="Y" targets="def1-4-19020113 t19020113-4-offence-1 t19020113-4-verdict-1"/> <p>(108) <persN

In [28]:
# We can save just the XML portion in a local file:
from pathlib import Path

directory = Path("C:/Users/swami/OneDrive/Attachments/Law/Lab_5_AI/data/old-bailey")
file_path = directory / f"old-bailey-{fifth_trial_id}.xml"

In [29]:


#directory.mkdir(parents=True, exist_ok=True)

#with open(file_path, 'w') as file:
 #   file.write(response.json()['hits']['hits'][0]['_source']['xml'])

In [30]:
# Now we'll get trial `t17031013-13` specifically, for the examples below, concerning a man named "Davis":
davis_trial_id = 't17031013-13'

url = f'https://www.dhi.ac.uk/api/data/oldbailey_record_single?idkey={davis_trial_id}'

response = requests.get(url, 
                        headers=headers)

response.raise_for_status()

trial_xml = response.json()['hits']['hits'][0]['_source']['xml']

# Save to file
with open(f'C:/Users/swami/OneDrive/Attachments/Law/Lab_5_AI/data/old-bailey-{davis_trial_id}.xml', 'w', encoding='utf-8') as file:
    file.write(trial_xml)
response.json()

{'took': 10,
 'timed_out': False,
 '_shards': {'total': 20, 'successful': 20, 'skipped': 0, 'failed': 0},
 'hits': {'total': 1,
  'max_score': 10.208518,
  'hits': [{'_index': 'dhids_oldbailey_record',
    '_type': 'doc',
    '_id': 'AYxEMK2rPld_y58Rz-bk',
    '_score': 10.208518,
    '_source': {'metadata': '<table xmlns:dhids="https://www.dhi.ac.uk/data/" class="table small"><tbody><tr><th scope="row">Text type</th><td>Trial account</td></tr><tr><th scope="row">Defendants</th><td>Samuel Davis</td></tr><tr><th scope="row">Offences</th><td><a href="../about/crimes#theft">Theft</a> &gt; <a href="../about/crimes#grandlarceny">Grand larceny</a></td></tr><tr><th scope="row">Session Date</th><td><a href="17031013#t17031013-13">13th October 1703</a></td></tr><tr><th scope="row">Reference Number</th><td>t17031013-13</td></tr><tr><th scope="row">Verdicts</th><td><a href="../about/verdicts#guilty">Guilty</a></td></tr><tr><th scope="row">Punishments</th><td><a href="../about/punishment#miscellan

In [31]:
trial_xml

'<div1 type="trialAccount" id="t17031013-13"> <interp inst="t17031013-13" type="prevdiv" value="t17031013-12" divtype="trialAccount"/> <interp inst="t17031013-13" type="nextdiv" value="t17031013-14" divtype="trialAccount"/> <interp inst="t17031013-13" type="div0" value="17031013" divtype="sessionsPaper"/> <interp inst="t17031013-13" type="collection" value="BAILEY"/> <interp inst="t17031013-13" type="year" value="1703"/> <interp inst="t17031013-13" type="uri" value="sessionsPapers/17031013"/> <interp inst="t17031013-13" type="date" value="17031013"/> <xptr imgpath="ob/1700s/17031013002.gif" imgtitle="Proceedings of the Old Bailey, 13th October ." imgrights="This image is reproduced courtesy of the British Library. Commercial use is prohibited without permission of the owner of the original." type="pageFacsimile" value="preceding" doc="17031013002"/> <join result="criminalCharge" id="t17031013-13-off60-c52" targOrder="Y" targets="t17031013-13-defend52 t17031013-13-off60 t17031013-13-ver

# Scraping all trials from 1902 - 1913

In [32]:
len(doc_ids)

9192

In [34]:
# for doc_id in doc_ids:
#     url = 'https://www.dhi.ac.uk/api/data/oldbailey_record_single?idkey={}'.format(doc_id)
    
#     try:
       
#         response = requests.get(url, 
#                                 headers=headers)  # Get the JSON with the User-Agent header
#         response.raise_for_status() # Check if request was successful
#         tree = response.json()
        
#         # Extract the XML content
#         xml_content = tree['hits']['hits'][0]['_source']['xml']
        
#         # Save the file with UTF-8 encoding
#         file_path = 'C:/Users/swami/OneDrive/Attachments/Law/Lab_5_AI/data/old-bailey-{}.xml'.format(doc_id)
#         with open(file_path, 'w', encoding='utf-8') as file:
#             file.write(xml_content)
            
#         print("Successfully saved: {}".format(doc_id))

#     except Exception as e:
#         print("Failed to download {}: {}".format(doc_id, e))
    
#     # 0.1-second pause to be a polite crawler
#     time.sleep(0.1)

In [37]:
#You can check if you saved the XML files by executing the cell below
import glob
from pathlib import Path

directory = Path("C:/Users/swami/OneDrive/Attachments/Law/Lab_5_AI/data")

xml_files = list(directory.glob("*.xml"))
xml_files

[WindowsPath('C:/Users/swami/OneDrive/Attachments/Law/Lab_5_AI/data/old-bailey-f19020113.xml'),
 WindowsPath('C:/Users/swami/OneDrive/Attachments/Law/Lab_5_AI/data/old-bailey-f19020210.xml'),
 WindowsPath('C:/Users/swami/OneDrive/Attachments/Law/Lab_5_AI/data/old-bailey-f19020310.xml'),
 WindowsPath('C:/Users/swami/OneDrive/Attachments/Law/Lab_5_AI/data/old-bailey-f19020407.xml'),
 WindowsPath('C:/Users/swami/OneDrive/Attachments/Law/Lab_5_AI/data/old-bailey-f19020505.xml'),
 WindowsPath('C:/Users/swami/OneDrive/Attachments/Law/Lab_5_AI/data/old-bailey-f19020602.xml'),
 WindowsPath('C:/Users/swami/OneDrive/Attachments/Law/Lab_5_AI/data/old-bailey-f19020630.xml'),
 WindowsPath('C:/Users/swami/OneDrive/Attachments/Law/Lab_5_AI/data/old-bailey-f19020721.xml'),
 WindowsPath('C:/Users/swami/OneDrive/Attachments/Law/Lab_5_AI/data/old-bailey-f19020909.xml'),
 WindowsPath('C:/Users/swami/OneDrive/Attachments/Law/Lab_5_AI/data/old-bailey-f19021020.xml'),
 WindowsPath('C:/Users/swami/OneDrive/At

## XML Syntax

First, we'll go over the syntax of a XML file. The basic unit of XML code is called an "element" or "node" and has a start and ending tag. The tags for each element look something like this:

 `<exampletag>some text</exampletag>`  

Run the next cell to look at the XML file of one of the cases from the OldBailey API!

In [38]:
# Samuel Davis trial from above
davis_trial_id = "t17031013-13"

example = requests.get(
    "https://www.dhi.ac.uk/api/data/oldbailey_record_single",
    params={"idkey": davis_trial_id},
    headers={"Accept": "application/json", "User-Agent": "Mozilla/5.0"}
).json()

example  

{'took': 3,
 'timed_out': False,
 '_shards': {'total': 20, 'successful': 20, 'skipped': 0, 'failed': 0},
 'hits': {'total': 1,
  'max_score': 10.208518,
  'hits': [{'_index': 'dhids_oldbailey_record',
    '_type': 'doc',
    '_id': 'AYxEMK2rPld_y58Rz-bk',
    '_score': 10.208518,
    '_source': {'metadata': '<table xmlns:dhids="https://www.dhi.ac.uk/data/" class="table small"><tbody><tr><th scope="row">Text type</th><td>Trial account</td></tr><tr><th scope="row">Defendants</th><td>Samuel Davis</td></tr><tr><th scope="row">Offences</th><td><a href="../about/crimes#theft">Theft</a> &gt; <a href="../about/crimes#grandlarceny">Grand larceny</a></td></tr><tr><th scope="row">Session Date</th><td><a href="17031013#t17031013-13">13th October 1703</a></td></tr><tr><th scope="row">Reference Number</th><td>t17031013-13</td></tr><tr><th scope="row">Verdicts</th><td><a href="../about/verdicts#guilty">Guilty</a></td></tr><tr><th scope="row">Punishments</th><td><a href="../about/punishment#miscellane

We want to parse XML, so let's separate that out--the code above that saved files locally did that, as you can see.

You can see that the key 'xml' is shortly after the 'idkey' that we used when we made the list of trial records above. Let's inspect the XML from Samuel Davis's trial.

In [39]:
# get just the xml from the Samuel Davis trial account
xml_data = example['hits']['hits'][0]['_source']['xml']
xml_data

'<div1 type="trialAccount" id="t17031013-13"> <interp inst="t17031013-13" type="prevdiv" value="t17031013-12" divtype="trialAccount"/> <interp inst="t17031013-13" type="nextdiv" value="t17031013-14" divtype="trialAccount"/> <interp inst="t17031013-13" type="div0" value="17031013" divtype="sessionsPaper"/> <interp inst="t17031013-13" type="collection" value="BAILEY"/> <interp inst="t17031013-13" type="year" value="1703"/> <interp inst="t17031013-13" type="uri" value="sessionsPapers/17031013"/> <interp inst="t17031013-13" type="date" value="17031013"/> <xptr imgpath="ob/1700s/17031013002.gif" imgtitle="Proceedings of the Old Bailey, 13th October ." imgrights="This image is reproduced courtesy of the British Library. Commercial use is prohibited without permission of the owner of the original." type="pageFacsimile" value="preceding" doc="17031013002"/> <join result="criminalCharge" id="t17031013-13-off60-c52" targOrder="Y" targets="t17031013-13-defend52 t17031013-13-off60 t17031013-13-ver

The `interp` tags at the beginning of the file are elements that don't have any plain text content. Note that elements may possibly be empty and not contain any text (i.e. `interp` elements mentioned earlier). If the element is empty, the tag may follow a format that looks similar to `<exampletag/>`, which is equivalent to `<exampletag></exampletag>`.

Elements may also contain other elements, which we call "children". Most children are indented, but the indents aren't necessary in XML and are used for clarity to show nesting. For example, if we go down to `<persName id="t17540116-4-defend46" type="defendantName">` , we see that the `rs` tag is a child of `persName`. We will explore about children in XML more in the next section. 

Lastly, elements may have attributes, which are in the format `<exampletag name_of_attribute="somevalue">`. Attributes are designed to store data related to a specific elements. Attributes **must** follow the quotes format (`name = "value"`). As you can tell, in this XML file, attributes are everywhere!

What was the verdict of this case? Was there a punsihment and if so, what was it?

List both and state whether you found it as plain text content or as an attribute.

## Using Beautiful Soup to parse XML
Now that we know what the syntax and structure of an XML file, let's figure out how to parse through one! We are going to load the same file from the second section and use [Beautiful Soup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) to navigate through elements in this file. 

First, we need to import the file into a Beautiful Soup instance. 

In [40]:
#from bs4 import BeautifulSoup # already imported 

xml_file = Path(f"C:/Users/swami/OneDrive/Attachments/Law/Lab_5_AI/data/old-bailey-{davis_trial_id}.xml")

xml_content = xml_file.read_text(encoding="utf-8")

soup = BeautifulSoup(xml_content, 
                    "xml")  # specify XML parser

print(soup.prettify())

<?xml version="1.0" encoding="utf-8"?>
<div1 id="t17031013-13" type="trialAccount">
 <interp divtype="trialAccount" inst="t17031013-13" type="prevdiv" value="t17031013-12"/>
 <interp divtype="trialAccount" inst="t17031013-13" type="nextdiv" value="t17031013-14"/>
 <interp divtype="sessionsPaper" inst="t17031013-13" type="div0" value="17031013"/>
 <interp inst="t17031013-13" type="collection" value="BAILEY"/>
 <interp inst="t17031013-13" type="year" value="1703"/>
 <interp inst="t17031013-13" type="uri" value="sessionsPapers/17031013"/>
 <interp inst="t17031013-13" type="date" value="17031013"/>
 <xptr doc="17031013002" imgpath="ob/1700s/17031013002.gif" imgrights="This image is reproduced courtesy of the British Library. Commercial use is prohibited without permission of the owner of the original." imgtitle="Proceedings of the Old Bailey, 13th October ." type="pageFacsimile" value="preceding"/>
 <join id="t17031013-13-off60-c52" result="criminalCharge" targOrder="Y" targets="t17031013-

In [41]:
body = soup.find('div1')
print(body.prettify())

<div1 id="t17031013-13" type="trialAccount">
 <interp divtype="trialAccount" inst="t17031013-13" type="prevdiv" value="t17031013-12"/>
 <interp divtype="trialAccount" inst="t17031013-13" type="nextdiv" value="t17031013-14"/>
 <interp divtype="sessionsPaper" inst="t17031013-13" type="div0" value="17031013"/>
 <interp inst="t17031013-13" type="collection" value="BAILEY"/>
 <interp inst="t17031013-13" type="year" value="1703"/>
 <interp inst="t17031013-13" type="uri" value="sessionsPapers/17031013"/>
 <interp inst="t17031013-13" type="date" value="17031013"/>
 <xptr doc="17031013002" imgpath="ob/1700s/17031013002.gif" imgrights="This image is reproduced courtesy of the British Library. Commercial use is prohibited without permission of the owner of the original." imgtitle="Proceedings of the Old Bailey, 13th October ." type="pageFacsimile" value="preceding"/>
 <join id="t17031013-13-off60-c52" result="criminalCharge" targOrder="Y" targets="t17031013-13-defend52 t17031013-13-off60 t1703101

In [42]:
soup.contents

[<div1 id="t17031013-13" type="trialAccount"> <interp divtype="trialAccount" inst="t17031013-13" type="prevdiv" value="t17031013-12"/> <interp divtype="trialAccount" inst="t17031013-13" type="nextdiv" value="t17031013-14"/> <interp divtype="sessionsPaper" inst="t17031013-13" type="div0" value="17031013"/> <interp inst="t17031013-13" type="collection" value="BAILEY"/> <interp inst="t17031013-13" type="year" value="1703"/> <interp inst="t17031013-13" type="uri" value="sessionsPapers/17031013"/> <interp inst="t17031013-13" type="date" value="17031013"/> <xptr doc="17031013002" imgpath="ob/1700s/17031013002.gif" imgrights="This image is reproduced courtesy of the British Library. Commercial use is prohibited without permission of the owner of the original." imgtitle="Proceedings of the Old Bailey, 13th October ." type="pageFacsimile" value="preceding"/> <join id="t17031013-13-off60-c52" result="criminalCharge" targOrder="Y" targets="t17031013-13-defend52 t17031013-13-off60 t17031013-13-ver

In [43]:
for child in body.children:
    if child.name:
        print(child.name, 
              child.get("type"), 
              child.get("value"))
choose_p = body.find("p")  # first "p" of the trial - note, that if you want the second <p> tag, 
                           # you'll need to use -> for child in body.find_all("p")[1].find_all(recursive=False) - findall"p"[1] would find the 2nd "p"

# loop over only named tags and show type/value attributes
for child in choose_p.find_all(recursive=False):  # recursive = false means only direct children - ie, dont go deeper into the tags (grandchildren)
    print(child.name, 
          child.get("type"), 
          child.get("value"))

interp prevdiv t17031013-12
interp nextdiv t17031013-14
interp div0 17031013
interp collection BAILEY
interp year 1703
interp uri sessionsPapers/17031013
interp date 17031013
xptr pageFacsimile preceding
join None None
p None None
p None None
persName defendantName None
placeName None None
interp placeName St. James Westminster
interp type defendantHome
join None None
interp defendantNameHome St. James Westminster
rs offenceDescription None
interp victims-institution no
interp victims-occupation Lady
interp victims-hisco-label Aristocrat
interp victims-hisco-code 1700
interp victims-hisco-class-1 0.5
interp victims-hisco-class-3 Upper (0.5-3)
persName victimName None
rs crimeDate None
join None None
rs occupation None
interp defendantNameLabel Coachman
join None None
rs verdictDescription None


This isn't very helpful, since we're still left with a bunch of tags and on top of that, we have a lot of repeating tags and names. Let's choose `placename` as our next tag and see what happens.

In [44]:
place_name = body.find("p").find("persName") # you can play around with the tags, for example persName

# iterate over its direct children
for child in place_name.find_all(recursive=True):
    print(child.name, 
          child.get("type"), 
          child.get("value"), 
          child.get_text(strip=True)) ## remove trailing Whitespace

interp surname Davis 
interp given Samuel 
interp gender male 


Nothing was printed, so it looks like we hit the end!

Note that, when `None` shows up, it usually means the tag exists, but it doesn’t have a `type` or `value` attribute. In other words:

The information is just in the text content of the tag, not in a structured attribute like `<interp>`.

So you have to look at `.string` or `.text` to get the actual text inside the tag.

Let's use `.string` to examine the data in this element, following the `.` path we used to get here.

In [45]:
body.p.placeName.string

'St. James Westminster'

 Find the defendant's name by traversing through the correct elements. You can check your answer with printed XML using `soup.contents`

You may find `body.p.persname.string` returns None. If a tag, `body.p.persname.string` in this case, contains more than one thing, then it’s not clear what .string should refer to, so .string is defined to be None. Which functions could help us locate the name instead?

In [46]:
soup = BeautifulSoup(xml_content,   ## just repeating what was above - ie where the soup comes from 
                     "xml")

#  Defendant 
defendant_tag = body.find("persName", {"type": "defendantName"})
defendant_name = f"{defendant_tag.find('interp', {'type':'given'})['value']} {defendant_tag.find('interp', {'type':'surname'})['value']}"
defendant_gender = defendant_tag.find("interp", {"type": "gender"})["value"]

# Defendant occupation (by id containing 'deflabel')
defendant_occupation_tag = body.find("rs", id=lambda x: x and "deflabel" in x)
defendant_occupation = defendant_occupation_tag.get_text(strip=True)

#  Victim 
victim_tag = body.find("persName", {"type": "victimName"})
victim_name = f"{victim_tag.find('interp', {'type':'given'})['value']} {victim_tag.find('interp', {'type':'surname'})['value']}"
victim_occupation_tag = victim_tag.find("rs", {"type": "occupation"})
victim_occupation = victim_occupation_tag.get_text(strip=True)

#  Offence 
offence_tag = body.find("rs", {"type": "offenceDescription"})
offence_text = offence_tag.get_text(strip=True)
offence_category = offence_tag.find("interp", {"type": "offenceCategory"})["value"]
offence_subcategory = offence_tag.find("interp", {"type": "offenceSubcategory"})["value"]

#  Verdict 
verdict_tag = body.find("rs", {"type": "verdictDescription"})
verdict_text = verdict_tag.get_text(strip=True)
verdict_category = verdict_tag.find("interp", {"type": "verdictCategory"})["value"]
plea = verdict_tag.find("interp", {"type": "plea"})["value"]

#  Punishment 
punishment_tag = body.find("rs", {"type": "punishmentDescription"})
punishment_subcategory = punishment_tag.find("interp", {"type": "punishmentSubcategory"})["value"]

#  Print all information 
print("Defendant:", defendant_name, f"({defendant_gender})")
print("Defendant occupation:", defendant_occupation)
print("Victim:", victim_name)
print("Victim occupation:", victim_occupation)
print("Offence:", offence_text)
print("Category:", offence_category, "/", offence_subcategory)
print("Verdict:", verdict_text)
print("Plea:", plea)
print("Punishment Subcategory:", punishment_subcategory)

Defendant: Samuel Davis (male)
Defendant occupation: Coachman
Victim: Catherine Herbert
Victim occupation: Lady
Offence: feloniously Stealing 58 Diamonds set in Silver gilt, value 250 l.
Category: theft / grandLarceny
Verdict: guilty
Plea: notGuilty
Punishment Subcategory: brandingOnCheek


In [47]:
# Extract main trial text (from first <p>) excluding structured tags
trial_text_tag = body.find("p")
trial_text_parts = []

for elem in trial_text_tag.children:
    if elem.name in ["rs", "interp", "join"]:  # skip structured tags
        continue
    if isinstance(elem, str):
        trial_text_parts.append(elem)
    else:
        trial_text_parts.append(elem.get_text())

# Join and normalize whitespace
trial_text = " ".join(" ".join(trial_text_parts).split())

trial_text

'Samuel Davis , of the Parish of St. James Westminster , was indicted for the Goods of the Honourable Catherine Lady Herbert , on the last. It appeared that the Jewels were put up in a Closet, which was lockt, and the Prisoner being a in the House, took his opportunity to take them; the Lady, when missing them, offered a Reward of Fourscore Pounds to any that could give any notice of it; upon enquiry, the Lady heard that a Diamond was sold on London-Bridge, and they described the Prisoner who sold it, and pursuing him, found the Prisoner at East-Ham, with all his Goods bundled up ready to be gone, and in his Trunk found all the Diamonds but one, which was found upon him in the Role of his Stocking, when searcht before the Justice. He denied the Fact, saying, He found them upon a great Heap of Rubbish, but could not prove it; and that being but a weak Excuse, the Jury found him .'


## Section 4: Putting it all in a dataframe

Now that we have a bunch of XML files and know how to parse through them to extract data, let's put the data from the XML files into a dataframe. Take a look at the XML for [this trial](https://www.dhi.ac.uk/api/data/oldbailey_record_single?idkey=t17031013-13) (and even better, look at what is or isn't consistent between that one and some others), and think about the structure of the data. How would you identify the people involved in a case? How would you identify their roles (witness/defendant/victim/other), or their genders? What can you learn about the alleged offence?

*Note:* Some cases have multiple defendants, multiple victims or multiple witnesses; however, most cases only have at most one of each. You can represent this in a dataframe by having $N$ columns for each property of a defendant, victim, etc., but this results in many many empty cells, and may not be amenable to analysis for the questions you come up with.

Think about the kinds of questions you may want to ask about this data, and refer to the XML for how you might answer them. For example, you may be interested in

- the words used specifically in describing the crime (notice that the text specifically between `<rs id="..." type="offenceDescription">` and `</rs>` gives you this)

- whether any victim was female

- whether any defendant was female

- the `category` (or `subCategory`) of the offense, etc.

- the entire text of the trial (sans tags)

These are questions that can be answered for most if not all cases, so they make good candidates for names of columns.

Consider the following function to get the `date` of the case and return it.

In [48]:
def case_date(soup):                       # Return the case date from a trial soup object
    date_tag = soup.find("interp", {"type": "date"})
    return date_tag.get("value") if date_tag else "Unknown"

print("Case", davis_trial_id, "happened on", case_date(soup))

Case t17031013-13 happened on 17031013


In [49]:
def people_in_case(case_soup):                      # Return a list of people in the trial with their metadata.
    people = []
    div1 = case_soup.find("div1")
    if not div1:
        return people

    for persName in div1.find_all('persName'):  # capital N
        person = {"type": persName.get("type")}
        for interp in persName.find_all('interp'):
            field_name = interp.get("type")
            field_value = interp.get("value")
            if field_name and field_value:
                person[field_name] = field_value
        people.append(person)
    return people

In [50]:
people = people_in_case(soup)
people

[{'type': 'defendantName',
  'surname': 'Davis',
  'given': 'Samuel',
  'gender': 'male'},
 {'type': 'victimName',
  'victimNameLabel': 'Lady',
  'surname': 'Herbert',
  'given': 'Catherine',
  'gender': 'female'}]

In [51]:
"""
Extract all <rs> elements from the case XML and return a dictionary.
Each key is the 'type' of <rs>, and the value is a dictionary with:
  - all <interp> fields as key-value pairs
  - 'text': the visible text inside the <rs> element
"""

def case_descriptions(case_soup):
    descriptions = {}
    for rs in case_soup.find_all('rs'):
        rs_type = rs.get('type', 'unknown')
        desc = {}
        for interp in rs.find_all('interp'):       # Collect interp metadata
            field_name = interp.get("type")
            field_value = interp.get("value")
            desc[field_name] = field_value  
        desc["text"] = rs.get_text(strip=True)      # Add visible text (strip leading/trailing whitespace)
        descriptions[rs_type] = desc
    return descriptions

In [52]:
descriptions = case_descriptions(soup)
descriptions

{'offenceDescription': {'offenceCategory': 'theft',
  'offenceSubcategory': 'grandLarceny',
  'text': 'feloniously Stealing 58 Diamonds set in Silver gilt, value 250 l.'},
 'occupation': {'text': 'Coachman'},
 'crimeDate': {'text': '28th of July'},
 'verdictDescription': {'verdictCategory': 'guilty',
  'verdictSubcategory': 'guiltyNoDetail',
  'plea': 'notGuilty',
  'text': 'guilty'},
 'punishmentDescription': {'punishmentCategory': 'miscPunish',
  'punishmentSubcategory': 'brandingOnCheek',
  'text': '[Branding. See summary.]'}}

In [53]:
pd.DataFrame([{"x": 1, "y": 10}, 
              {"x": 12, "y": 111, "z": 999}])

,x,y,z
0,1,10,NaN
1,12,111,999.0


In [54]:
from math import nan

def table_of_cases(xml_file_names):
    rows = []

    for xml_file in xml_file_names:                          # Load and parse XML
        with open(f'C:/Users/swami/OneDrive/Attachments/Law/Lab_5_AI/data/old-bailey-{xml_file}.xml', "rb") as f:
            case_soup = BeautifulSoup(f, "xml")

        #  Map occupations via joins 
        occupation_map = {}
        for join_tag in case_soup.find_all("join", {"result": "persNameOccupation"}):
            targets = join_tag.attrs.get("targets", "").split()
            if len(targets) >= 2:
                pers_id = targets[0]                        # persName ID
                rs_tag = case_soup.find(id=targets[1])
                if rs_tag and rs_tag.attrs.get("type") == "occupation":
                    occupation_map[pers_id] = rs_tag.get_text(strip=True)

        #  Extract people 
        people = []
        for persName in case_soup.find_all('persName'):
            person = {"type": persName.attrs.get("type")}                                              
            for interp in persName.find_all('interp'):          # interp metadata
                field = interp.attrs.get("type")
                value = interp.attrs.get("value", "")
                person[field] = value
            # Full name
            given = person.get("given", "")
            surname = person.get("surname", "")
            person["fullName"] = f"{given} {surname}".strip()
            # Occupation from join mapping
            person["occupation"] = occupation_map.get(persName.attrs.get("id"), "")
            people.append(person)

        #  Extract case info 
        date = case_date(case_soup)
        descriptions = case_descriptions(case_soup)
        case_id = case_soup.div1.attrs.get("id", nan)

        # Initialize row
        row = {
            "date": date,
            "id": case_id,
            "text": " ".join(case_soup.get_text(separator=" ").split()),  # cleaned full text
            "any_defendant_female": False,
            "any_defendant_male": False,
            "any_victim_female": False,
            "any_victim_male": False,
            # Offence
            "offenceText": nan,
            "offenceCategory": nan,
            "offenceSubcategory": nan,
            # Verdict
            "verdictText": nan,
            "verdictCategory": nan,
            # Punishment
            "punishmentText": nan,
            "punishmentCategory": nan,
            "punishmentSubcategory": nan,
            # Names and occupations
            "defendantNames": nan,
            "defendantOccupations": nan,
            "victimNames": nan,
            "victimOccupations": nan
        }

        #  Extract <rs> info 
        offence = descriptions.get("offenceDescription", {})
        row["offenceText"] = offence.get("text", nan)
        row["offenceCategory"] = offence.get("offenceCategory", nan)
        row["offenceSubcategory"] = offence.get("offenceSubcategory", nan)

        verdict = descriptions.get("verdictDescription", {})
        row["verdictText"] = verdict.get("text", nan)
        row["verdictCategory"] = verdict.get("verdictCategory", nan)

        punishment = descriptions.get("punishmentDescription", {})
        row["punishmentText"] = punishment.get("text", nan)
        row["punishmentCategory"] = punishment.get("punishmentCategory", nan)
        row["punishmentSubcategory"] = punishment.get("punishmentSubcategory", nan)

        #  Collect gender flags, names, and occupations 
        defendants = [p for p in people if p.get("type")=="defendantName"]
        victims = [p for p in people if p.get("type")=="victimName"]

        # Gender flags
        for d in defendants:
            g = d.get("gender", "").lower()
            if g == "female":
                row["any_defendant_female"] = True
            elif g == "male":
                row["any_defendant_male"] = True

        for v in victims:
            g = v.get("gender", "").lower()
            if g == "female":
                row["any_victim_female"] = True
            elif g == "male":
                row["any_victim_male"] = True

        # Names and occupations as comma-separated strings
        def join_nonempty(items):
            items = [i.strip() for i in items if i and i.strip()]
            return ", ".join(items) if items else nan

        row["defendantNames"] = join_nonempty([d.get("fullName","") for d in defendants])
        row["defendantOccupations"] = join_nonempty([d.get("occupation","") for d in defendants])
        row["victimNames"] = join_nonempty([v.get("fullName","") for v in victims])
        row["victimOccupations"] = join_nonempty([v.get("occupation","") for v in victims])
        
        rows.append(row)

    return pd.DataFrame(rows)


In [59]:
df = table_of_cases(doc_ids[:])
df.head()

,date,id,text,any_defendant_female,any_defendant_male,any_victim_female,any_victim_male,offenceText,offenceCategory,offenceSubcategory,verdictText,verdictCategory,punishmentText,punishmentCategory,punishmentSubcategory,defendantNames,defendantOccupations,victimNames,victimOccupations
0,19020113,f19020113,CENTRAL CRIMINAL COURT Sessions Paper. DIMSDAL...,False,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,19020113,t19020113-105,"105. WALTER HEATH (82) , PLEADED GUILTY to ste...",False,True,False,True,"stealing £13, the money ofGeorge Whitehead, hi...",theft,stealingFromMaster,PLEADED GUILTY,guilty,Eighteen months' hard labour.,imprison,hardLabour,WALTER HEATH,NaN,"GEORGE WHITEHEAD, HENRY JOHN MANNING",NaN
2,19020113,t19020113-2,"(106) THOMAS GEORGE WAKEFIELD , to forging and...",False,True,False,False,", to forging and uttering an undertaking for t...",deception,forgery,[Pleaded guilty: See original trial image.],guilty,Six months' in the Second Division.,imprison,imprisonNoDetail,THOMAS GEORGE WAKEFIELD,NaN,NaN,NaN
3,19020113,t19020113-3,"(107) FREDERICK JOHN RIDGWELL (29) , to steali...",False,True,False,False,", to stealing, while employed under the Post O...",theft,mail,[Pleaded guilty: See original trial image.],guilty,Nine months' hard labour.,imprison,hardLabour,FREDERICK JOHN RIDGWELL,NaN,NaN,NaN
4,19020113,t19020113-4,"(108) WALTER LACEY (38) , to forging an author...",False,True,False,False,", to forging an authority for the withdrawal o...",deception,forgery,[Pleaded guilty: See original trial image.],guilty,Three months' in the Second Division.,imprison,imprisonNoDetail,WALTER LACEY,NaN,NaN,NaN


In [60]:
df.shape

(9192, 19)

In [61]:
df.columns

Index(['date', 'id', 'text', 'any_defendant_female', 'any_defendant_male',
       'any_victim_female', 'any_victim_male', 'offenceText',
       'offenceCategory', 'offenceSubcategory', 'verdictText',
       'verdictCategory', 'punishmentText', 'punishmentCategory',
       'punishmentSubcategory', 'defendantNames', 'defendantOccupations',
       'victimNames', 'victimOccupations'],
      dtype='object')

In [62]:
df.describe()

,defendantOccupations,victimOccupations
count,0.0,0.0
mean,NaN,NaN
std,NaN,NaN
min,NaN,NaN
25%,NaN,NaN
50%,NaN,NaN
75%,NaN,NaN
max,NaN,NaN


In [63]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9192 entries, 0 to 9191
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   9192 non-null   object 
 1   id                     9192 non-null   object 
 2   text                   9192 non-null   object 
 3   any_defendant_female   9192 non-null   bool   
 4   any_defendant_male     9192 non-null   bool   
 5   any_victim_female      9192 non-null   bool   
 6   any_victim_male        9192 non-null   bool   
 7   offenceText            9053 non-null   object 
 8   offenceCategory        9053 non-null   object 
 9   offenceSubcategory     9053 non-null   object 
 10  verdictText            9006 non-null   object 
 11  verdictCategory        9006 non-null   object 
 12  punishmentText         7404 non-null   object 
 13  punishmentCategory     7404 non-null   object 
 14  punishmentSubcategory  7404 non-null   object 
 15  defe

In [64]:
df.iloc[0,2]

"CENTRAL CRIMINAL COURT Sessions Paper. DIMSDALE, MAYOR. THIRD SESSION, HELD JANUARY 13TH, 1902. MINUTES OF EVIDENCE, TAKEN IN SHORT-HAND BY JAMES DROVER BARNETT AND ALEXANDER BUCKLER , Short-hand Writers to the Court, ROLLS CHAMBERS , No. 89, CHANCERY LANE. LONDON: STEVENS AND SONS, LIMITED, 119, CHANCERY LANE. Law Booksellers and Publishers. THE WHOLE PROCEEDINGS On the King's Commission of OYER AND TERMINER AND GAOL DELIVERY FOR The City of London, AND GAOL DELIVERY FOR THE COUNTY OF MIDDLESEX AND THE PARTS OF THE COUNTIES OF ESSEX, KENT, AND SURREY WITHIN THE JURISDICTION OF THE CENTRAL CRIMINAL COURT, Held on Monday, January 13th, 1902, and following days, Before the Right Hon. SIR JOSEPH COCKFIELD DIMSDALE , Knt., M.P., LORD MAYOR of the City of London; the Rt. Hon. LORD ALVERSTONE , G.C.M.G., Lord Chief Justice of England; the Hon. Sir ARTHUR RICHARD JELF , one other of the Justices of His Majesty's High Court; Sir HENRY EDMUND KNIGHT , Knt.; Sir JOSEPH RENALS , Bart., and Sir J

In [65]:
df.to_csv("C:/Users/swami/OneDrive/Attachments/Law/Lab_5_AI/data/old-bailey/data.csv", index=False)

In [66]:
def get_clean_counts(series):
    return (
        series
        .dropna()
        .astype(str)
        .str.split(',')
        .explode()
        .str.strip()
        .replace('', pd.NA)
        .dropna()
        .value_counts()
    )
print("Defendant Occupations Counts:")
print(get_clean_counts(df['defendantOccupations']))

print("\nVictim Occupations Counts:")
print(get_clean_counts(df['victimOccupations']))

Defendant Occupations Counts:
Series([], Name: count, dtype: int64)

Victim Occupations Counts:
Series([], Name: count, dtype: int64)


In [67]:
df.head()

,date,id,text,any_defendant_female,any_defendant_male,any_victim_female,any_victim_male,offenceText,offenceCategory,offenceSubcategory,verdictText,verdictCategory,punishmentText,punishmentCategory,punishmentSubcategory,defendantNames,defendantOccupations,victimNames,victimOccupations
0,19020113,f19020113,CENTRAL CRIMINAL COURT Sessions Paper. DIMSDAL...,False,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,19020113,t19020113-105,"105. WALTER HEATH (82) , PLEADED GUILTY to ste...",False,True,False,True,"stealing £13, the money ofGeorge Whitehead, hi...",theft,stealingFromMaster,PLEADED GUILTY,guilty,Eighteen months' hard labour.,imprison,hardLabour,WALTER HEATH,NaN,"GEORGE WHITEHEAD, HENRY JOHN MANNING",NaN
2,19020113,t19020113-2,"(106) THOMAS GEORGE WAKEFIELD , to forging and...",False,True,False,False,", to forging and uttering an undertaking for t...",deception,forgery,[Pleaded guilty: See original trial image.],guilty,Six months' in the Second Division.,imprison,imprisonNoDetail,THOMAS GEORGE WAKEFIELD,NaN,NaN,NaN
3,19020113,t19020113-3,"(107) FREDERICK JOHN RIDGWELL (29) , to steali...",False,True,False,False,", to stealing, while employed under the Post O...",theft,mail,[Pleaded guilty: See original trial image.],guilty,Nine months' hard labour.,imprison,hardLabour,FREDERICK JOHN RIDGWELL,NaN,NaN,NaN
4,19020113,t19020113-4,"(108) WALTER LACEY (38) , to forging an author...",False,True,False,False,", to forging an authority for the withdrawal o...",deception,forgery,[Pleaded guilty: See original trial image.],guilty,Three months' in the Second Division.,imprison,imprisonNoDetail,WALTER LACEY,NaN,NaN,NaN
